<a href="https://colab.research.google.com/github/swabhimansahu2004/Hybrid-TinyML-Environmental-Monitoring/blob/Swabhiman_Teacher_Model/Phase_4_Pruning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf
import tensorflow_model_optimization as tfmot
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 1. Re-Load and Split Data
df = pd.read_csv('smoke_detection_iot.csv')
features = ['Humidity[%]', 'Pressure[hPa]', 'Raw H2', 'NC2.5', 'NC1.0',
            'PM2.5', 'eCO2[ppm]', 'PM1.0', 'NC0.5', 'Temperature[C]',
            'TVOC[ppb]', 'Raw Ethanol']
X = df[features]
y = df['Fire Alarm']

X_train_full, X_test, y_train_full, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train_full, y_train_full, test_size=0.125, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

# 2. MANUALLY RECONSTRUCT Student_A Architecture
# This avoids the "InputLayer" config error
model_to_prune = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(len(features),)),
    tf.keras.layers.Dense(16, activation='relu'),
    tf.keras.layers.Dense(8, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

# 3. Load only the weights from your saved file
# This bypasses the corrupted metadata and just takes the 'intelligence'
model_to_prune.load_weights('best_distilled_student.h5')

print("✅ Model reconstructed and weights loaded successfully!")

✅ Model reconstructed and weights loaded successfully!


In [ ]:
# 1. Define the Pruning Schedule
# We remove weights over 10 epochs using Polynomial Decay
batch_size = 32
epochs = 10
end_step = np.ceil(len(X_train) / batch_size).astype(np.int32) * epochs

pruning_params = {
      'pruning_schedule': tfmot.sparsity.keras.PolynomialDecay(
          initial_sparsity=0.0,
          final_sparsity=0.50, # Target: 50% of weights become zero
          begin_step=0,
          end_step=end_step
      )
}

# 2. Wrap the model
model_for_pruning = tfmot.sparsity.keras.prune_low_magnitude(model_to_prune, **pruning_params)

# 3. Re-compile (Mandatory to activate the pruning logic)
model_for_pruning.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

print("✂️ Pruning Wrapper Applied. Ready for training.")
model_for_pruning.summary()

✂️ Pruning Wrapper Applied. Ready for training.
Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 prune_low_magnitude_dense   (None, 16)                402       
 (PruneLowMagnitude)                                             
                                                                 
 prune_low_magnitude_dense_  (None, 8)                 266       
 1 (PruneLowMagnitude)                                           
                                                                 
 prune_low_magnitude_dense_  (None, 1)                 19        
 2 (PruneLowMagnitude)                                           
                                                                 
Total params: 687 (2.70 KB)
Trainable params: 353 (1.38 KB)
Non-trainable params: 334 (1.32 KB)
_________________________________________________________________


In [3]:
# The UpdatePruningStep callback is required during training
callbacks = [
    tfmot.sparsity.keras.UpdatePruningStep(),
]

print("📡 Starting Iterative Pruning (10 Epochs)...")
model_for_pruning.fit(
    X_train, y_train,
    batch_size=32,
    epochs=10,
    validation_data=(X_val, y_val),
    callbacks=callbacks,
    verbose=1
)

# Verify accuracy hasn't crashed
_, pruned_acc = model_for_pruning.evaluate(X_test, y_test, verbose=0)
print(f"\n✅ Surgery Finished!")
print(f"Original Student Accuracy: 99.88%")
print(f"Pruned Student Accuracy:   {pruned_acc*100:.2f}%")

📡 Starting Iterative Pruning (10 Epochs)...
Epoch 1/10
1371/1371 [==============================] - 13s 5ms/step - loss: 0.0044 - accuracy: 0.9987 - val_loss: 0.0063 - val_accuracy: 0.9986
Epoch 2/10
1371/1371 [==============================] - 4s 3ms/step - loss: 0.0102 - accuracy: 0.9986 - val_loss: 0.0035 - val_accuracy: 0.9989
Epoch 3/10
1371/1371 [==============================] - 4s 3ms/step - loss: 0.0055 - accuracy: 0.9984 - val_loss: 0.0133 - val_accuracy: 0.9987
Epoch 4/10
1371/1371 [==============================] - 4s 3ms/step - loss: 0.0070 - accuracy: 0.9985 - val_loss: 0.0052 - val_accuracy: 0.9979
Epoch 5/10
1371/1371 [==============================] - 4s 3ms/step - loss: 0.0076 - accuracy: 0.9978 - val_loss: 0.0060 - val_accuracy: 0.9984
Epoch 6/10
1371/1371 [==============================] - 4s 3ms/step - loss: 0.0260 - accuracy: 0.9952 - val_loss: 0.0104 - val_accuracy: 0.9981
Epoch 7/10
1371/1371 [==============================] - 4s 3ms/step - loss: 0.0125 - accura

In [8]:
import zipfile
import tempfile
# 1. Remove the pruning wrappers to get a standard Keras model back
final_pruned_model = tfmot.sparsity.keras.strip_pruning(model_for_pruning)

# 2. Save the model
final_pruned_model.save('student_pruned.h5')

# 3. Compare File Sizes
import os
def get_gzipped_model_size(file):
    # Zip the file because pruning creates 'zeros' which compress extremely well
    import zipfile
    _, zipped_file = tempfile.mkstemp('.zip')
    with zipfile.ZipFile(zipped_file, 'w', compression=zipfile.ZIP_DEFLATED) as f:
        f.write(file)
    return os.path.getsize(zipped_file)

print("\n📊 MEMORY SAVINGS REPORT")
print(f"Size of Distilled Model: {os.path.getsize('best_distilled_student.h5') / 1024:.2f} KB")
print(f"Size of Pruned Model:    {os.path.getsize('student_pruned.h5') / 1024:.2f} KB")
print(f"Zipped Pruned Model (Ready for ESP32): {get_gzipped_model_size('student_pruned.h5') / 1024:.2f} KB")


📊 MEMORY SAVINGS REPORT
Size of Distilled Model: 23.30 KB
Size of Pruned Model:    17.53 KB
Zipped Pruned Model (Ready for ESP32): 2.50 KB
